### 라이브러리

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance

### STEP1 전처리 결과 불러오기 및 분리

In [2]:
def load_preprocessed(
    path,
    target_col='Y_Quality',
    class_col='Y_Class',
    id_cols=['PRODUCT_ID', 'PRODUCT_CODE'],
    time_cols='TIMESTAMP'
):
    df = pd.read_csv(path)
    y = df[target_col].copy()
    y_class = df[class_col].copy()
    temp = df[id_cols + [time_cols]].copy()
    X = df.drop(columns=[target_col, class_col] + id_cols + [time_cols])
    return X, y, y_class, temp

### STEP2 CV 기반 Feature Importance 계산

In [3]:
def compute_cv_importance(X, y_class, n_splits=5, n_repeats=10, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    importances = pd.DataFrame(index=X.columns)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y_class.iloc[train_idx], y_class.iloc[val_idx]

        model = lgb.LGBMClassifier(
            random_state=random_state,
            class_weight='balanced',
            verbose=-1
        )
        model.fit(X_train, y_train)

        result = permutation_importance(
            model, X_val, y_val,
            n_repeats=n_repeats, random_state=random_state, n_jobs=-1
        )
        importances[f"fold_{fold}"] = result.importances_mean

    importances["mean_importance"] = importances.mean(axis=1)
    return importances.sort_values("mean_importance", ascending=False)

### STEP3 중요도 기준 컬럼 선택

In [4]:
def select_features(importance_df, threshold=0.0, top_n=None):
    selected = importance_df[importance_df["mean_importance"] > threshold]
    if top_n is not None:
        selected = selected.head(top_n)
    return selected.index.tolist()

### 전체 피처 선택 함수

In [5]:
def select_by_importance(
    path,
    n_splits=5,
    n_repeats=10,
    threshold=0.0,
    top_n=None,
    random_state=42
):
    X, y, y_class, temp = load_preprocessed(path)
    importance = compute_cv_importance(X, y_class, n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)
    selected_cols = select_features(importance, threshold=threshold, top_n=top_n)

    print("=" * 50)
    print(f"{path} : {X.shape[1]}개 중 {len(selected_cols)}개 선택")
    print("=" * 50)

    return X[selected_cols], y, y_class, temp, importance

### 피처 선택 적용

In [6]:
X_A, y_A, yclass_A, temp_A, importance_A = select_by_importance("A_31_preprocessed.csv")
X_TO, y_TO, yclass_TO, temp_TO, importance_TO = select_by_importance("TO_31_preprocessed.csv")

A_31_preprocessed.csv : 551개 중 93개 선택


TO_31_preprocessed.csv : 334개 중 43개 선택


### CSV 형식으로 저장

In [7]:
df_A_selected = pd.concat([X_A, y_A, yclass_A, temp_A], axis=1)
df_A_selected.to_csv("A_31_selected.csv", index=False, encoding="utf-8-sig")

df_TO_selected = pd.concat([X_TO, y_TO, yclass_TO, temp_TO], axis=1)
df_TO_selected.to_csv("TO_31_selected.csv", index=False, encoding="utf-8-sig")